```markdown
# Desvendando o Melhor Agrupamento (Best Cluster)

Este notebook é um guia prático projetado para ensinar como comparar diferentes algoritmos de Aprendizado Não Supervisionado (*Unsupervised Learning*) para encontrar a melhor estrutura de agrupamento (*clustering*) em um conjunto de dados. Aqui, você aprenderá a preparar os dados, executar múltiplos algoritmos (K-Means, Agrupamento Hierárquico e DBSCAN) e avaliar cientificamente qual deles apresenta o melhor desempenho.

### 🎯 Objetivos de Aprendizagem
Ao final deste estudo, você será capaz de:
1. **Compreender o papel da padronização** de dados antes de aplicar algoritmos baseados em distância.
2. **Implementar e parametrizar** três dos principais algoritmos de clustering: K-Means, Agglomerative Clustering e DBSCAN.
3. **Utilizar o coeficiente de Silhueta (*Silhouette Score*)** como métrica quantitativa para comparar a qualidade de diferentes agrupamentos.
4. **Analisar criticamente os resultados** para selecionar a configuração algorítmica ideal para o seu problema.
```

In [ ]:
import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score

```markdown
### 📦 1. Preparando as Ferramentas (Bibliotecas)

Antes de começarmos a construir nossos modelos, precisamos importar as ferramentas que farão o trabalho pesado por nós. Em Python, fazemos isso importando bibliotecas específicas:

*   **`numpy` (Numerical Python):** Essencial para manipulação de matrizes e operações matemáticas de alta performance. Nós a usaremos para gerar intervalos numéricos flexíveis para os parâmetros do DBSCAN.
*   **`pandas`:** Funciona como uma planilha extremamente poderosa. Usaremos para organizar, visualizar e ordenar os resultados dos nossos testes em uma tabela dinâmica.
*   **`sklearn` (Scikit-Learn):** A biblioteca definitiva para Machine Learning clássico em Python. Dela extraímos:
    *   `datasets`: Para carregar um conjunto de dados de teste real (o famoso dataset Iris).
    *   `StandardScaler`: Uma ferramenta de preparação para colocar todas as nossas variáveis na mesma escala matemática.
    *   `KMeans`, `AgglomerativeClustering`, `DBSCAN`: Os três algoritmos de agrupamento que iremos testar e comparar.
    *   `silhouette_score`: Nossa métrica de avaliação, que mede o quão bem definidos e separados estão os nossos grupos.
```

In [ ]:
def compare_algorithms(X, max_clusters):
  results = []
  cluster_range = range(2, max_clusters +1)

  #kmeans
  for n_clusters in cluster_range:
    kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init='auto')
    clusters = kmeans.fit_predict(X)
    silhouette_avg = silhouette_score(X, clusters)
    results.append(('KMeans',n_clusters,silhouette_avg))

  #agglomerativo
  for n_clusters in cluster_range:
    agglomerative = AgglomerativeClustering(n_clusters=n_clusters)
    clusters = agglomerative.fit_predict(X)
    silhouette_avg = silhouette_score(X, clusters)
    results.append(('Agglomerative',n_clusters,silhouette_avg))

  #dbscan
  eps_values = np.arange(0.1,0.9,0.1)
  for eps in eps_values:
    dbscan = DBSCAN(eps=eps, min_samples=5)
    clusters = dbscan.fit_predict(X)
    if len(set(clusters)) > 1:
      silhouette_avg = silhouette_score(X, clusters)
      results.append(('DBSCAN',eps,silhouette_avg))

  return results


```markdown
### ⚙️ 2. A Função de Comparação de Algoritmos

**O que este passo faz:**
Criamos uma função chamada `compare_algorithms` que automatiza o teste dos três algoritmos sob diferentes configurações (hiperparâmetros) e calcula o desempenho de cada um.

**Por que ele é necessário?**
Em Machine Learning, raramente sabemos de antemão qual algoritmo ou qual número de grupos funcionará melhor. Em vez de escrever códigos repetitivos para testar cada cenário manualmente, criamos esta função para rodar uma "maratona de testes" (Grid Search) de forma limpa e organizada.

**Como funciona a lógica por trás:**
1.  **K-Means:** Testamos diferentes números de clusters (de 2 até o limite definido por `max_clusters`).
2.  **Agglomerative (Hierárquico):** Também testamos o mesmo intervalo de grupos.
3.  **DBSCAN:** Como ele não exige que definamos o número de grupos de antemão (ele descobre sozinho com base na densidade), nós variamos o parâmetro `eps` (o raio de busca da vizinhança) de `0.1` a `0.8`. Só calculamos a Silhueta se ele encontrar pelo menos 2 grupos (evitando erro caso classifique tudo como ruído ou um único grupo).

> 💡 **Para pensar:** No K-Means, o parâmetro `n_init='auto'` foi definido para evitar avisos de depreciação de versões futuras da biblioteca, garantindo que o algoritmo inicialize de forma eficiente e automática. No entanto, lembre-se de que o K-Means é sensível à inicialização dos centroides; o `random_state=0` garante que nossos resultados sejam reprodutíveis.
```

In [ ]:
iris = datasets.load_iris()
scaler = StandardScaler()
scaled_data = scaler.fit_transform(iris.data)

results = compare_algorithms(scaled_data, 10)
df = pd.DataFrame(results, columns=['Agrupador','Clusters','Score'])
df


,Agrupador,Clusters,Score
0,KMeans,2.0,0.581750
1,KMeans,3.0,0.459948
2,KMeans,4.0,0.386941
3,KMeans,5.0,0.345511
4,KMeans,6.0,0.343719
5,KMeans,7.0,0.329236
6,KMeans,8.0,0.335194
7,KMeans,9.0,0.352488
8,KMeans,10.0,0.348607
9,Agglomerative,2.0,0.577035


```markdown
### 🌸 3. Carregando, Padronizando e Executando os Testes

**O que este passo faz:**
1. Carrega o clássico banco de dados **Iris** (que contém medições físicas de 3 espécies de flores de Íris).
2. Aplica o `StandardScaler` nos dados das flores.
3. Executa a nossa função de comparação e exibe uma tabela estruturada com as pontuações de cada teste.

**Por que ele é necessário?**
*   **Padronização (Scaling):** Algoritmos de clustering medem distâncias geométricas entre os pontos. Se uma variável tiver números muito grandes (ex: peso em gramas) e outra tiver números muito pequenos (ex: largura em centímetros), a variável de escala maior vai dominar o cálculo injustamente. O `StandardScaler` deixa todas com média 0 e desvio padrão 1, garantindo equilíbrio.
*   **Métrica de Silhueta (*Silhouette Score*):** Essa pontuação varia de **-1 a +1**.
    *   Valores próximos a **+1** significam que os pontos estão muito próximos dos seus respectivos grupos e muito distantes dos grupos vizinhos (excelente agrupamento).
    *   Valores próximos a **0** indicam sobreposição de grupos.
    *   Valores **negativos** indicam que os pontos provavelmente foram atribuídos ao grupo errado.

**Como interpretar a tabela de saída:**
Ao analisar a tabela gerada, você pode comparar diretamente o desempenho. Por exemplo, vemos que o **K-Means com 2 grupos** obteve um score de `0.581750` (alto), enquanto o **DBSCAN com `eps=0.3`** obteve um score negativo de `-0.194195` (o que significa que essa distância foi muito pequena para a densidade dos dados, gerando grupos ruins ou excesso de ruído).
```

In [ ]:
max_score_index = df['Score'].idxmax()
print(df.loc[max_score_index])

Agrupador     KMeans
Clusters         2.0
Score        0.58175
Name: 0, dtype: object


```markdown
### 🏆 4. Encontrando o Vencedor Absoluto

**O que este passo faz:**
Localiza automaticamente a linha na nossa tabela de resultados que possui o maior valor de *Silhouette Score* e imprime essas informações na tela.

**Por que ele é necessário?**
Em conjuntos de dados reais, podemos gerar centenas de combinações de testes. Olhar linha por linha seria exaustivo e propenso a falhas humanas. Usar a função `idxmax()` do Pandas nos dá de forma rápida e precisa a melhor configuração absoluta encontrada.

**Como interpretar a saída:**
O resultado aponta o **K-Means com 2 Clusters** como o vencedor absoluto da competição para esta parametrização, atingindo um score de aproximação de `0.58175`.

> 💡 **Para pensar:** Sabendo que o dataset Iris possui originalmente **3 espécies** reais de flores, por que você acha que o algoritmo encontrou um melhor resultado matemático separando-os em apenas **2 grupos**?
> *Dica de professor:* Duas das três espécies de flores Iris são biologicamente muito parecidas entre si em suas dimensões e se sobrepõem muito no espaço geométrico, fazendo com que, para os algoritmos matemáticos puramente geométricos, agrupá-las juntas em um único bloco faça mais sentido do que tentar separá-las em duas.
```

```markdown
---

## 🚨 Erros Comuns e Dicas de Sobrevivência

1.  **Esquecer de Padronizar os Dados (`StandardScaler`):** Se você tentar rodar algoritmos baseados em distância (como K-Means e Hierárquico) em dados não normalizados, o resultado final pode ser completamente distorcido por colunas que possuam escalas numéricas maiores.
2.  **A Armadilha do DBSCAN (`eps` muito pequeno ou muito grande):** Diferente dos outros, o DBSCAN pode classificar dados como ruído (marcando-os com o label `-1`). Se o `eps` for muito pequeno (como `0.1`), quase todos os pontos viram ruído e o cálculo da silhueta pode falhar ou dar negativo. Se for muito grande, todos os dados entram em um único grupo gigante.
3.  **Não fixar o `random_state`:** O K-Means escolhe pontos iniciais aleatórios. Se você não fixar o `random_state`, o seu código poderá dar um resultado ligeiramente diferente a cada vez que for executado.

---

## 📝 Resumo da Aula

*   **Não Supervisionado:** Aprendemos a agrupar dados sem precisar de respostas corretas prévias (*labels*).
*   **Padronização é Lei:** Entendemos que a distância geométrica exige que os dados estejam na mesma escala.
*   **A Tríade dos Algoritmos:**
    *   **K-Means:** Rápido, mas exige definição prévia de $K$ e assume que os grupos são esféricos.
    *   **Agglomerative:** Constrói uma árvore de agrupamento de baixo para cima.
    *   **DBSCAN:** Encontra grupos com base em densidade e formas complexas, ignorando ruídos (outliers).
*   **Métrica de Silhueta:** Nosso termômetro para validar numericamente a harmonia interna dos clusters formados.

---

## 🏋️ Exercícios Propostos

1.  **Exercício 1 (Fácil):** Altere o limite superior de clusters testados (`max_clusters`) na chamada da função de `10` para `15`. O algoritmo ideal muda ou o K-Means com 2 grupos continua sendo o campeão?
2.  **Exercício 2 (Médio):** O DBSCAN usa como padrão a distância Euclidiana. O algoritmo `AgglomerativeClustering` permite mudar a métrica de distância (parâmetro `metric`) e o tipo de ligação (`linkage`). Tente pesquisar na documentação do Scikit-Learn e implementar uma modificação na função para testar o agrupamento hierárquico usando a ligação completa (`linkage='complete'`).
3.  **Exercício 3 (Desafiador):** Crie uma plotagem gráfica (usando `matplotlib` ou `seaborn`) do tipo gráfico de linha (*lineplot*) mostrando o comportamento do Silhouette Score para o K-Means conforme o número de clusters aumenta de 2 a 10. Você consegue visualizar graficamente o "ponto de quebra" ou método do cotovelo?

---

## 🚀 Para ir Além

Para expandir seus conhecimentos em engenharia de dados e machine learning, pesquise sobre os seguintes tópicos:
*   **Método do Cotovelo (Elbow Method):** Outra forma visual muito famosa de determinar o melhor número de clusters para o K-Means baseado na inércia.
*   **Maldição da Dimensionalidade:** Entenda por que calcular distâncias fica muito difícil e impreciso quando nossos dados possuem muitas colunas (dimensões).
*   **Algoritmo HDBSCAN:** Uma evolução moderna do DBSCAN que consegue encontrar agrupamentos de densidades variáveis de forma automática.
```